In [5]:
import pandas as pd
df = pd.read_csv("data/sample_transactions.csv")

df["date"] = pd.to_datetime(df["date"])
df["date"].dtype

df["date"].dt.month

pd.to_datetime(df["date"], errors="coerce")

test = pd.Series(["2026-01-01", "not a date"])
pd.to_datetime(test, errors="coerce")

test = pd.DataFrame({
    "date": ["2026-01-01", "not a date", "2026-01-03"],
    "amount": [100, 200, "abc"]
})

test["date"] = pd.to_datetime(test["date"], errors="coerce")
test["amount"] = pd.to_numeric(test["amount"], errors="coerce")
test

test["date"].isna()

bad = test["date"].isna() | test["amount"].isna()
bad

test[bad]     # the rejected rows
test[~bad]    # the good rows

test2 = pd.DataFrame({
    "description": ["Swiggy", "  ", None, "Netflix"],
    "amount": [-400, -100, -50, -649]
})

test2["description"] = test2["description"].str.strip()
test2

missing = test2["description"].isna() | (test2["description"] == "")
missing

test2.loc[missing, "description"] = "(no description)"
test2

test3 = pd.DataFrame({
    "date": ["2026-09-12", "2026-09-12", "2026-09-13"],
    "description": ["Swiggy Order", "Swiggy Order", "Swiggy Order"],
    "amount": [-450, -450, -450]
})

test3.duplicated(subset=["date", "description", "amount"])

is_dup = test3.duplicated(subset=["date", "description", "amount"])
test3[~is_dup]

is_dup.sum()



np.int64(1)

In [2]:
import os
os.chdir(r"C:\Users\Hari Prasath\moneymap")


In [7]:
from src.data_loader import load_transactions
from src.data_cleaning import clean_transactions

raw = load_transactions("data/sample_transactions.csv")
clean, rejected, summary = clean_transactions(raw)
summary


rules = {
    "swiggy": "Food Delivery",
    "zomato": "Food Delivery",
    "netflix": "Subscriptions",
}

desc = "Random Shop"

for keyword, category in rules.items():
    if keyword in desc.lower():
        print(category)

In [16]:
def categorize(desc):
    for keyword, category in rules.items():
        if keyword in desc.lower():
            return category
    return "Uncategorized"

categorize("Swiggy Order")
categorize("Zomato Order")
categorize("random shop")

clean["category"] = clean["description"].apply(categorize)
clean.head(10)

clean["category"].value_counts()

rules = {
    "refund": "Refunds",
    "credit card payment": "Credit Card Payment",
    "own savings": "Internal Transfer",
    "salary": "Salary",
    "freelance": "Freelance Income",
    "rent": "Housing",
    "swiggy": "Food Delivery",
    "zomato": "Food Delivery",
    "netflix": "Subscriptions",
    "spotify": "Subscriptions",
    "groceries": "Groceries",
    "uber": "Transportation",
    "petrol": "Fuel",
    "electricity": "Utilities",
    "amazon": "Shopping",
    "myntra": "Shopping",
    "udemy": "Education",
    "pvr": "Entertainment",
    "pharmacy": "Healthcare",
    "clinic": "Healthcare",
    "indigo": "Travel",
}

clean["category"] = clean["description"].apply(categorize)
clean.head(10)

clean["category"].value_counts()

income = clean[clean["amount"] > 0]["amount"].sum()
expenses = clean[clean["amount"] < 0]["amount"].sum()
income, expenses

(income + expenses) / income * 100

np.float64(18.357244318181817)

In [22]:
transfers = clean["category"] == "Internal Transfer"
refunds = clean["category"] == "Refunds"

# leave out transfers and refunds
money = clean[~transfers & ~refunds]

income = money[money["amount"] > 0]["amount"].sum()
expenses = money[money["amount"] < 0]["amount"].sum()

# refunds reduce expenses
expenses = expenses + clean[refunds]["amount"].sum()

income, expenses

(income + expenses) / income * 100

cc = clean["category"] == "Credit Card Payment"
cc_total = clean[cc]["amount"].sum()

expenses_without_cc = expenses - cc_total   # minus a negative = adds back
(income + expenses_without_cc) / income * 100

from src.analytics import calculate_summary

calculate_summary(clean)

calculate_summary(clean, count_card_payments=False)

{'income': np.int64(140000),
 'expenses': np.int64(77653),
 'net_cash_flow': np.int64(62347),
 'savings_rate': np.float64(44.53357142857143)}

In [1]:
import os
os.chdir(r"C:\Users\Hari Prasath\moneymap")
from src.data_loader import load_transactions
from src.data_cleaning import clean_transactions
from src.categorization import add_categories
from src.analytics import calculate_summary

raw = load_transactions("data/sample_transactions.csv")
clean, rejected, summary = clean_transactions(raw)
clean = add_categories(clean)

calculate_summary(clean)

spend = clean[(clean["amount"] < 0) & (clean["category"] != "Internal Transfer")]

by_cat = spend.groupby("category")["amount"].sum().abs().sort_values(ascending=False)
by_cat


import plotly.express as px
import pandas as pd

fig = px.bar(
    by_cat.reset_index(),
    x="amount",
    y="category",
    orientation="h",
    title="Spending by category",
)
fig.update_layout(yaxis={"categoryorder": "total ascending"})
fig.show()

clean["month"] = clean["date"].dt.to_period("M").astype(str)
clean[["date", "month"]].head()

rows = []
for month, group in clean.groupby("month"):
    result = calculate_summary(group)
    rows.append({"month": month, "income": result["income"], "expenses": result["expenses"]})

monthly = pd.DataFrame(rows)
monthly

import plotly.express as px

fig = px.bar(
    monthly,
    x="month",
    y=["income", "expenses"],
    barmode="group",
    title="Monthly income vs expenses",
)
fig.show()

def pct_change(old, new):
    if old == 0:
        return None
    return (new - old) / old * 100

print(pct_change(1000, 1250))
print(pct_change(0, 500))
print(pct_change(1000, 800))

clean["month"] = clean["date"].dt.to_period("M").astype(str)

spend = clean[(clean["amount"] < 0) & (clean["category"] != "Internal Transfer")]

table = spend.pivot_table(
    index="category",
    columns="month",
    values="amount",
    aggfunc="sum",
).abs().fillna(0)

table

changes = {}
for category in table.index:
    old = table.loc[category, "2026-08"]
    new = table.loc[category, "2026-09"]
    changes[category] = pct_change(old, new)

pd.Series(changes)

from src.analytics import category_changes

category_changes(clean, "2026-08", "2026-09")


from src.data_loader import load_transactions
from src.data_cleaning import clean_transactions
from src.categorization import add_categories
from src.analytics import calculate_summary, category_changes

raw = load_transactions("data/sample_transactions.csv")
clean, rejected, summary = clean_transactions(raw)
clean = add_categories(clean)


spend = clean[(clean["amount"] < 0) & (clean["category"] != "Internal Transfer")]
by_cat = spend.groupby("category")["amount"].sum().abs().sort_values(ascending=False)

share = by_cat / by_cat.sum() * 100
share.round(1)
share.head(3).sum()

from src.analytics import category_shares

share, top3 = category_shares(clean)
print(top3)
share.round(1)

25.0
None
-20.0
67.38166938461076


category
Housing                36.0
Credit Card Payment    21.5
Groceries               9.9
Shopping                8.3
Food Delivery           5.2
Travel                  4.8
Fuel                    4.7
Utilities               3.8
Subscriptions           2.3
Healthcare              1.6
Transportation          0.8
Entertainment           0.7
Education               0.5
Name: amount, dtype: float64